# Data Cleaning Làm sạch dữ liệu và Kỹ thuật đặc trưng
**STAT3013 | TrainHyp**

| Mục | Chi tiết |
|-----|---------|
| Input  | Dữ liệu thô ban đầu |
| Output | `data_features.csv` |

## Lý do khoa học cho việc làm sạch dữ liệu
> **Mục tiêu của bước này không phải là để chạy code không bị lỗi, mà là để đảm bảo tính toàn vẹn và độ tin cậy của dữ liệu phân tích.**

Ý nghĩa của việc làm sạch và chọn lọc:
- **Tính nhất quán:** Chỉ giữ lại các quan sát thực sự liên quan đến mục tiêu nghiên cứu (Hypertrophy), loại bỏ các dữ liệu nhiễu.
- **Tiêu chuẩn hóa:** Tính toán và chuẩn hóa biến mục tiêu (hedges_g) để tạo ra thước đo thống nhất, giúp các mô hình và phép kiểm định sau này đánh giá chính xác mức độ hiệu quả của phương pháp tập luyện.
- **Nguyên tắc "Garbage in, garbage out":** Bất kỳ mô hình thống kê nào cũng sẽ cho kết quả sai lệch nếu dữ liệu đầu vào chứa nhiều thông tin rác hoặc không được phân loại đúng cách.

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as snsS
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Load raw data
df_raw = pd.read_excel("/content/gdrive/MyDrive/STAT3013/Dataset/Extracted-Data.xlsx")
print(f"Raw shape: {df_raw.shape}")
print(f"Columns: {df_raw.shape[1]}")
df_raw["outcome"].value_counts()

Raw shape: (710, 95)
Columns: 95


,count
outcome,
Strength,490
Hypertrophy,220


## 1. Filter - Chỉ giữ Hypertrophy

In [4]:
# [TIỀN XỬ LÝ]: Cô lập biến mục tiêu (Target Filtering).
# Chỉ giữ lại các nghiên cứu đo lường "Hypertrophy" (Phì đại cơ bắp) để đảm bảo tính đồng nhất (homogeneity) cho mô hình thống kê, tránh nhiễu từ các nghiên cứu về Strength hay Endurance.
df = df_raw[df_raw["outcome"] == "Hypertrophy"].copy()

# Làm mới chỉ mục (Reset Index) để tránh lỗi lệch dòng (Alignment) khi thực hiện các phép toán vector hoặc nối bảng ở các bước sau.
df = df.reset_index(drop=True)

print(f"After filter: {df.shape[0]} rows (từ {df_raw.shape[0]} rows)")
print(f"Outcome values: {df['outcome'].unique()}")

After filter: 220 rows (từ 710 rows)
Outcome values: ['Hypertrophy']


## 2. Chọn cột (29 cột từ 95 cột)

In [5]:
# [CẤU TRÚC HÓA]: Phân nhóm biến số theo các miền giá trị (Domain-specific grouping).
# Việc phân loại rõ ràng giúp quản lý thuộc tính (Feature Management) dễ dàng hơn khi thực hiện phân tích đa nhân tố.

# Thông tin định danh và thiết kế nghiên cứu.
STUDY_INFO = [
    "number", "first.author", "year", "design", "upper.lower"
]

# Đặc điểm nhân trắc học của đối tượng tham gia (Demographics).
SUBJECT = [
    "age", "sex.male", "train.status", "body.mass", "years.exp"
]

# Các biến số độc lập về giao thức tập luyện (Independent Variables).
TRAINING = [
    "sets.week.all", "sets.week.direct",
    "frequency.direct", "sessions.per.week",
    "reps.week.all", "rep.range.all",
    "interset.rest.min.all", "percentage.failure.all",
    "failure.nonfailure.mixed.direct", "weeks"
]

# Biến điều tiết về dinh dưỡng (Moderators).
NUTRITION = [
    "nutrition.controlled.surplus?", "nutrition.controlled.deficit?"
]

# [LƯU Ý]: Nhóm biến kỹ thuật dùng để tính toán Hedges' g (Target), sẽ được loại bỏ trước khi đưa vào các mô hình Machine Learning để tránh hiện tượng rò rỉ dữ liệu (Data Leakage).
COMPUTE_G = [
    "pre.mean", "post.mean", "pre.sd", "post.sd", "n",
    "measure", "outcome"
]

# [THỰC THI]: Tinh lọc Dataset, loại bỏ các cột dư thừa để tối ưu bộ nhớ và tăng tốc độ xử lý.
ALL_COLS = STUDY_INFO + SUBJECT + TRAINING + NUTRITION + COMPUTE_G
df = df[ALL_COLS].copy()

print(f"Selected: {len(ALL_COLS)} columns")
print(f"Shape: {df.shape}")

Selected: 29 columns
Shape: (220, 29)


## 3. Kiểm tra missing values

In [6]:
# [KIỂM TOÁN DỮ LIỆU]: Thống kê tỷ lệ dữ liệu khuyết thiếu (Missing Data Analysis).
# [ĐIỂM CHẠM HỌC THUẬT]: Việc xác định % Missing giúp quyết định chiến lược xử lý: Loại bỏ hàng (Drop) nếu tỷ lệ thấp, hoặc dùng Imputation (điền khuyết) nếu biến đó có tầm quan trọng cao về mặt sinh lý học nhưng dữ liệu không đầy đủ.
miss = (df.isnull().sum() / len(df) * 100).round(1).sort_values(ascending=False)
miss_df = pd.DataFrame({"Column": miss.index, "Missing %": miss.values})

# [KỸ THUẬT]: Chỉ tập trung hiển thị các cột "có vấn đề" (> 0%) để nhanh chóng nhận diện các lỗ hổng thông tin trong dataset nghiên cứu.
print(miss_df[miss_df["Missing %"] > 0].to_string(index=False))

                         Column  Missing %
                      years.exp       45.0
                         pre.sd       10.0
                        post.sd       10.0
                      body.mass        4.5
         percentage.failure.all        4.5
          interset.rest.min.all        3.6
                       sex.male        2.7
failure.nonfailure.mixed.direct        1.8


## 4. Xử lí missing values

In [7]:
# [KỸ THUẬT]: Ép kiểu dữ liệu (Type Casting).
# Chuyển đổi các cột có nguy cơ lẫn tạp chất chuỗi (string) về dạng số, dùng errors="coerce" để biến các giá trị không hợp lệ thành NaN.
NUMERIC_COLS = [
    "years.exp", "body.mass", "sex.male",
    "rep.range.all", "interset.rest.min.all", "percentage.failure.all"
]
for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# [CHIẾN LƯỢC IMPUTATION]: Điền khuyết bằng Median cho các biến nhân trắc học và tập luyện.
# Median được chọn thay vì Mean để bảo vệ mô hình khỏi tác động của các giá trị ngoại lai (Outliers) trong nghiên cứu y sinh.
df["years.exp"] = df["years.exp"].fillna(df["years.exp"].median())
df["body.mass"] = df["body.mass"].fillna(df["body.mass"].median())

# [CHUẨN HÓA]: Đưa tỷ lệ giới tính (sex.male) về thang đo [0, 1].
# Điều này giúp các thuật toán ML (như KNN, Linear Regression) không bị "lệch" trọng số do sự chênh lệch scale giữa các biến.
df["sex.male"] = df["sex.male"].fillna(df["sex.male"].median()) / 100.0

# Xử lý các biến tập luyện còn lại.
# Với percentage.failure, ta giả định các nghiên cứu không báo cáo là không tập đến ngưỡng thất bại (0%).
df["rep.range.all"] = df["rep.range.all"].fillna(df["rep.range.all"].median())
df["interset.rest.min.all"] = df["interset.rest.min.all"].fillna(df["interset.rest.min.all"].median())
df["percentage.failure.all"] = df["percentage.failure.all"].fillna(0)

# [MỤC ĐÍCH]: Dùng Mode (Yếu vị) cho biến phân loại để giữ tính nhất quán của dữ liệu gốc.
mode_fail = df["failure.nonfailure.mixed.direct"].mode()[0]
df["failure.nonfailure.mixed.direct"] = df["failure.nonfailure.mixed.direct"].fillna(mode_fail)

# Kiểm tra cuối: Đảm bảo Dataset đã "sạch" hoàn toàn để sẵn sàng cho Modeling.
print(f"Missing sau impute:")
remaining = df.isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else "Không còn missing!")

Missing sau impute:
pre.sd     22
post.sd    22
dtype: int64


## 5. Tính Hedges'g (Target variable)

In [8]:
def compute_hedges_g(row):
    """
    [MỤC ĐÍCH]: Tính toán Hedges' g - Chỉ số đo lường độ lớn hiệu ứng (Effect Size) chuẩn hóa.
    [LÝ DO SỬ DỤNG]: So với Cohen's d, Hedges' g hiệu quả hơn trong các tập mẫu nhỏ (n < 20) bằng cách áp dụng hệ số hiệu chỉnh J, giúp giảm sai số hệ thống (bias).
    """
    try:
        pre_m, post_m = row["pre.mean"], row["post.mean"]
        pre_sd, post_sd = row["pre.sd"], row["post.sd"]
        n = row["n"]

        # Rào chắn dữ liệu: Đảm bảo đủ các thành phần tham số để tính toán.
        if any(pd.isna(v) for v in [pre_m, post_m, pre_sd, post_sd, n]):
            return np.nan

        # [TOÁN HỌC]: Tính Pooled SD - Độ lệch chuẩn gộp để đại diện cho biến thiên tổng thể của nghiên cứu.
        pooled = np.sqrt((pre_sd**2 + post_sd**2) / 2)
        if pooled == 0: return np.nan

        # Tính Cohen's d (biên độ thay đổi thô).
        d = (post_m - pre_m) / pooled

        # [HỆ SỐ HIỆU CHỈNH J]: Thành phần cốt lõi của Hedges' g nhằm "phạt" các nghiên cứu có cỡ mẫu quá thấp.
        J = 1 - 3 / (4 * (2 * n - 2) - 1)

        return round(d * J, 4)
    except:
        return np.nan

# Thực thi tính toán theo từng dòng (Row-wise operation).
df["hedges_g"] = df.apply(compute_hedges_g, axis=1)

# [VỆ SINH DỮ LIỆU]: Loại bỏ triệt để các nghiên cứu không đủ thông tin tính toán Effect Size.
# Việc này đảm bảo biến mục tiêu (Target) luôn sạch trước khi tiến hành phân tích thống kê suy diễn.
n_before = len(df)
df = df.dropna(subset=["hedges_g"]).reset_index(drop=True)
n_after = len(df)

print(f"Hedges g computed: {n_after} / {n_before} rows")
print(f"Hedges g stats:")
print(df["hedges_g"].describe().round(4))

Hedges g computed: 198 / 220 rows
Hedges g stats:
count    198.0000
mean       0.4291
std        0.3698
min       -0.2080
25%        0.1454
50%        0.3602
75%        0.6146
max        2.2588
Name: hedges_g, dtype: float64


## 6. Loại outliers(|g| > 3)

In [9]:
# [KIỂM SOÁT NHIỄU]: Loại bỏ các giá trị ngoại lai (Outliers) cực đoan.
# [LÝ DO SINH LÝ]: Trong nghiên cứu thể hình, giá trị Hedges' g > 3 thường được coi là bất thường (do lỗi đo lường hoặc kịch bản nghiên cứu không thực tế).
# Việc giới hạn biên độ giúp mô hình tập trung vào các xu hướng tăng trưởng khả thi, tránh bị chệch hướng bởi các điểm dữ liệu quá cá biệt.
n_before = len(df)
df = df[df["hedges_g"].abs() <= 3].reset_index(drop=True)
n_after = len(df)

# Báo cáo kết quả vệ sinh dữ liệu: Đảm bảo tập dữ liệu cuối cùng có dải giá trị (range) ổn định, sẵn sàng cho các phép kiểm định t-test và ANOVA.
print(f"Outliers removed: {n_before - n_after} rows")
print(f"Final shape: {df.shape}")
print(f"hedges_g range: [{df['hedges_g'].min():.4f}, {df['hedges_g'].max():.4f}]")

Outliers removed: 0 rows
Final shape: (198, 30)
hedges_g range: [-0.2080, 2.2588]


## 7. Feature Engineering

In [10]:
# [KỸ THUẬT]: Rời rạc hóa (Discretization) biến tổng số hiệp tập (Volume).
# [LÝ DO NGHIỆP VỤ]: Trong nghiên cứu Sports Science, việc chia nhóm giúp xác định các ngưỡng (Thresholds) hiệu quả.
# Quy tắc phân loại: Low (Dưới 6 hiệp - Duy trì), Moderate (7-15 hiệp - Tối ưu), High (Trên 15 hiệp - Chuyên sâu).
def vol_cat(x):
    if x <= 6:   return "Low"
    elif x <= 15: return "Moderate"
    else:         return "High"

df["volume_category"] = df["sets.week.all"].apply(vol_cat)

# [CẤU TRÚC HÓA]: Ép kiểu Categorical với thuộc tính Ordered=True.
# Điều này cực kỳ quan trọng để các biểu đồ (Boxplot/Bar chart) tự động hiển thị theo thứ tự tiến trình Low -> Moderate -> High thay vì sắp xếp theo bảng chữ cái.
df["volume_category"] = pd.Categorical(df["volume_category"],
                           categories=["Low","Moderate","High"], ordered=True)

print("volume_category:")
print(df["volume_category"].value_counts().sort_index())

volume_category:
volume_category
Low         48
Moderate    61
High        89
Name: count, dtype: int64


In [11]:
# [KỸ THUẬT]: Mã hóa nhị phân (Binary Encoding) cho biến ngưỡng thất bại.
# [MỤC ĐÍCH]: Chuyển đổi dữ liệu định danh (Failure/Non-failure) sang dạng số (1/0) để máy tính có thể thực hiện các phép toán thống kê.
# Phân loại: 1 (Tập đến ngưỡng thất bại hoàn toàn), 0 (Các trường hợp còn lại: Mixed hoặc Non-failure).
def fail_bin(x):
    if str(x).lower() == "failure": return 1
    return 0

df["failure_binary"] = df["failure.nonfailure.mixed.direct"].apply(fail_bin)

# Kiểm tra phân phối mẫu: Đảm bảo có đủ dữ liệu ở cả hai nhóm để thực hiện kiểm định T-test hoặc Chi-square sau này.
print("failure_binary:")
print(df["failure_binary"].value_counts())

failure_binary:
failure_binary
1    159
0     39
Name: count, dtype: int64


In [12]:
# [KỸ THUẬT]: Hợp nhất các biến điều kiện (Feature Consolidation).
# [LÝ DO NGHIỆP VỤ]: Dinh dưỡng là biến điều tiết (Moderator) cực kỳ quan trọng.
# Việc gộp "Surplus" (Thừa calo) và "Deficit" (Thâm hụt calo) vào một cột giúp dễ dàng quan sát bức tranh tổng thể về môi trường đồng hóa/dị hóa của đối tượng nghiên cứu.
def nutr_cat(row):
    s = str(row["nutrition.controlled.surplus?"]).strip().lower()
    d = str(row["nutrition.controlled.deficit?"]).strip().lower()
    if s == "yes": return "Surplus"
    if d == "yes": return "Deficit"
    return "None"

df["nutrition_category"] = df.apply(nutr_cat, axis=1)
print("nutrition_category:")
print(df["nutrition_category"].value_counts())

# [TỐI ƯU CHO ML]: Tạo biến nhị phân "has_nutrition_control".
# Trong Machine Learning, việc biết "có kiểm soát dinh dưỡng hay không" thường quan trọng hơn là chỉ biết loại hình (do số mẫu Surplus/Deficit có thể bị lệch).
# Biến Flag này giúp mô hình đánh giá tác động của tính kỷ luật trong ăn uống đối với kết quả phì đại cơ bắp.
df["has_nutrition_control"] = (
    df["nutrition_category"].isin(["Surplus", "Deficit"])
).astype(int)

print(f"\n[FIX] has_nutrition_control:")
print(df["has_nutrition_control"].value_counts())

nutrition_category:
nutrition_category
None       188
Surplus      6
Deficit      4
Name: count, dtype: int64

[FIX] has_nutrition_control:
has_nutrition_control
0    188
1     10
Name: count, dtype: int64


## 8. Chuẩn hóa dtype

In [13]:
# [KỸ THUẬT]: Mã hóa thứ bậc (Ordinal Encoding) cho trình độ tập luyện.
# [LÝ DO NGHIỆP VỤ]: Trình độ tập luyện có tính kế thừa (Untrained < Recreational < Trained).
# Việc gán số (0, 1, 2) giúp mô hình Machine Learning hiểu được "thứ hạng" của đối tượng, từ đó đánh giá chính xác hơn khả năng thích nghi cơ bắp giảm dần khi trình độ tăng cao.
train_map = {"untrained": 0, "recreational": 1, "trained": 2}
df["train_status_enc"] = df["train.status"].str.lower().map(train_map).fillna(1).astype(int)

# [MÃ HÓA NHỊ PHÂN]: Chuyển đổi vùng cơ thể (Upper/Lower) thành biến Dummy.
df["upper_body"] = (df["upper.lower"].str.lower() == "upper").astype(int)

# [KỸ THUẬT]: Rời rạc hóa biến mục tiêu (Labeling Target).
# Dựa trên ngưỡng của Cohen để phân loại mức độ phì đại. Việc chuyển từ hồi quy (Regression) sang phân loại (Classification) giúp bài toán trở nên thực tiễn hơn khi cần dự đoán khả năng tăng trưởng ở mức Thấp, Trung bình hay Cao.
def hyp_class(g):
    if g < 0.2:   return "Low"
    elif g < 0.8: return "Medium"
    else:         return "High"

df["hyp_class"] = df["hedges_g"].apply(hyp_class)
df["hyp_class"] = pd.Categorical(df["hyp_class"],
                    categories=["Low","Medium","High"], ordered=True)

# [TỐI ƯU HIỂN THỊ]: Làm sạch định dạng số (Data Precision Control).
# Giới hạn 2 chữ số thập phân cho các biến liên tục để báo cáo trở nên gọn gàng, đồng thời tránh sai số tích lũy không đáng có do độ chính xác quá cao của dấu phẩy động.
INT_COLS = ["number","year","n","failure_binary",
            "has_nutrition_control","train_status_enc","upper_body"]
for col in df.columns:
    if col not in INT_COLS and df[col].dtype in ["float64","float32"]:
        df[col] = df[col].round(2)

print("hyp_class (ML classification target):")
print(df["hyp_class"].value_counts().sort_index())
print(f"[FIX] Float columns làm tròn 2 chữ số thập phân")

hyp_class (ML classification target):
hyp_class
Low        60
Medium    112
High       26
Name: count, dtype: int64
[FIX] Float columns làm tròn 2 chữ số thập phân


## 9. Lưu file output

In [14]:
# [MỤC ĐÍCH]: Lưu trữ dữ liệu sau khi đã làm sạch và chuẩn hóa hoàn toàn.
# [KỸ THUẬT]: Loại bỏ cột 'nutrition_category' do tỷ lệ khuyết thiếu quá cao (~95%), tránh gây nhiễu cho các phân tích định lượng, thay thế bằng biến nhị phân 'has_nutrition_control' đã tạo trước đó.
df_save = df.drop(columns=["nutrition_category"], errors="ignore")
df_save.to_csv("data_cleaned.csv", index=False)
print(f"Saved data_cleaned.csv: {df_save.shape}")

# [MỤC ĐÍCH]: Khởi tạo tập dữ liệu đặc trưng (Feature Set) dành riêng cho bài toán Machine Learning.
# [CHIẾN LƯỢC]: Tập hợp các biến đã được mã hóa (Encoded) và xử lý ngoại lai.
# Cấu trúc gồm: 15 biến đầu vào (Inputs) phục vụ dự đoán và 2 biến mục tiêu (Targets - Hedges' g cho Regression & Hyp_class cho Classification).
FEATURES = [
    "sets.week.all", "sets.week.direct",
    "frequency.direct", "sessions.per.week",
    "reps.week.all", "rep.range.all",
    "interset.rest.min.all", "percentage.failure.all",
    "failure_binary", "weeks",
    "train_status_enc", "sex.male", "age", "upper_body",
    "has_nutrition_control",
    "hedges_g", "hyp_class"
]
df_feat = df[FEATURES].copy()
df_feat.to_csv("data_features.csv", index=False)
print(f"Saved data_features.csv: {df_feat.shape}")

# [TỔNG KẾT QUY TRÌNH]: Báo cáo sự hao hụt dữ liệu qua các bước (Data Attrition Summary).
# Việc minh bạch các con số từ 710 hàng xuống còn ~200 hàng cho thấy sự nghiêm túc trong việc loại bỏ "rác", đảm bảo tính khách quan cho đồ án STAT3013.
print(f"{'='*50}")
print(f"FINAL SUMMARY:")
print(f"  Raw rows:        710")
print(f"  After filter:    220 (Hypertrophy only)")
print(f"  After hedges_g:  {df.shape[0]} rows usable")
print(f"  Features:        {len(FEATURES)-2} input + 2 targets")
print(f"  Volume Low:      {(df['volume_category']=='Low').sum()}")
print(f"  Volume Moderate: {(df['volume_category']=='Moderate').sum()}")
print(f"  Volume High:     {(df['volume_category']=='High').sum()}")

Saved data_cleaned.csv: (198, 36)
Saved data_features.csv: (198, 17)
FINAL SUMMARY:
  Raw rows:        710
  After filter:    220 (Hypertrophy only)
  After hedges_g:  198 rows usable
  Features:        15 input + 2 targets
  Volume Low:      48
  Volume Moderate: 61
  Volume High:     89
